# Empathy dataset: original message → empathetic rewrite

This experiment follows a **parallel rewrite design**, like the informal/formal pairing described for GYAFC. Each pair contains the **same speaker's same message**, expressed neutrally and more empathetically. It is not a user question paired with an assistant's answer.

**Illustrative pair (not teacher-generated):**

- Original: “Your workload is heavy, but I need the draft by 3 PM.”
- Empathetic rewrite: “I understand your workload is heavy; I still need the draft by 3 PM.”

The request, deadline, speaker, recipient, and stance remain fixed. The target adds acknowledgement of a circumstance already present in the message. It must not invent sadness, promise help, apologize on the speaker's behalf, soften a refusal into agreement, or add advice.

The teacher receives a **fixed original message** and returns it unchanged as `low`, with an empathetic rewrite as `high`. The notebook checks that `low` exactly matches the original. That gives us original/rewritten pairs rather than two freely generated replies.

The output is a small synthetic pilot, not an established empathy benchmark. The first sections generate, inspect and export 50 pairs. The final sections feed them into our existing Qwen all-layer extractor and build a separate website catalog for testing. The normal website catalog remains unchanged. Use the repository `.venv` kernel. Keep API keys in environment variables.

In [19]:
from pathlib import Path
import csv, json, os, sys
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'pyproject.toml').exists() and (p / 'src/secondthought').exists()), None)
assert ROOT is not None, 'Open this notebook inside the SecondThought repository.'
os.chdir(ROOT)
for path in (ROOT, ROOT / 'src'):
    if str(path) not in sys.path: sys.path.insert(0, str(path))

from secondthought.lab.synthetic import (
    FeatureDefinition, make_plan, load_plan, generate, prepare, sha, read_jsonl,
)
RUN = ROOT / 'data/synthetic/empathy-rewrite-50-v1'
print('Pairs and generation records will be saved in:', RUN)

Pairs and generation records will be saved in: /Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1


## 1. Fixed source messages and pair rules

The 50 hand-authored originals below contain requests, feedback, boundaries, and responses to explicitly stated feelings. These are **messages to rewrite**, not prompts to answer. Empathy is relational: allowing a recipient's stated circumstance in the original makes acknowledgement possible without guessing facts.

For this pilot:

- Keep the original immutable; generate only its more empathetic expression.
- Preserve facts, numbers, requested actions, negation, certainty, and who does what.
- Keep grammar, politeness, and formality comparable. Neutral must not mean rude.
- Acknowledge only circumstances or feelings explicitly in the original. Do not add help, promises, apologies, advice, or agreement.
- Allow modest extra wording for acknowledgement, bounded by a 1.6 word-count ratio. Review verbosity as a possible confound.

Scenario families are split before generation. Variants of an original stay in the same split. Start with one rewrite per original; expand independent messages after reviewing the pilot. Changing these inputs requires a fresh `RUN`/version.

The 50-family split is **30 extraction / 10 held-out ordering / 10 reserved test**. Existing generated responses from the 30-message pilot are reused only when their exact teacher prompts match. Original response IDs and file hashes remain recorded. This is a small integration pilot; ten held-out pairs give coarse layer colours, not a reliable ranking.

In [20]:
FEATURE = dict(
    id='empathy', version='rewrite-v1', label='Empathy',
    low=('Copy original_message EXACTLY, preserving every word, capitalization and punctuation. '
         'This is a fixed neutral message to rewrite, not a prompt to answer. Never generate a new low endpoint.'),
    high=('Rewrite original_message in the SAME speaker voice, for the SAME recipient, with more '
          'empathic acknowledgement of only the feelings or circumstances explicitly present in the original. '
          'Keep its facts, requested actions, refusals, deadlines, certainty and practical content unchanged. '
          'Do not answer it, invent feelings, add advice, offer help, apologize, promise, or agree to anything new.'),
    preserve=['original low endpoint verbatim', 'speaker and recipient', 'facts and numbers',
              'requested actions', 'negation and refusals', 'certainty', 'deadlines', 'practical meaning'],
    nuisance_controls=['similar length', 'same grammar quality', 'same formality', 'same politeness',
                       'no extra practical content', 'no unsupported emotional inference'],
    min_words=6, max_words=70, max_word_ratio=1.6, scale='unit_l2')

# Fixed source text. Add more independent originals here; avoid near-duplicate families.
# Required terms are mechanical checks, not substitutes for semantic review.
CASES = [
 ('draft', 'Your workload is heavy, but I need the draft by 3 PM.', ['3 PM', 'draft']),
 ('meeting', 'You are worried about the meeting. Let us review the agenda together.', ['agenda']),
 ('notes', 'You feel overlooked because your contribution is missing from the notes. Ask for it to be added.', ['notes']),
 ('attachment', 'You are embarrassed about the wrong attachment. Send the correct file to Alex.', ['Alex']),
 ('capacity', 'You are already busy, so I will ask someone else to take this task.', ['task']),
 ('interview', 'You are disappointed about the interview result. You can ask for feedback.', ['feedback']),
 ('new_team', 'You feel out of place on the new team. You can introduce yourself to one teammate.', ['teammate']),
 ('revision', 'You are discouraged by the feedback, but this draft still needs one more revision.', ['revision']),
 ('promotion', 'You are excited about the promotion. Ask your manager about the new responsibilities.', ['manager']),
 ('exam', 'You studied hard and are disappointed with the result. Review the marked answers.', ['answers']),
 ('assignment', 'The first question is confusing you. Ask the instructor to explain it.', ['instructor']),
 ('course', 'You are proud of finishing the course after a difficult year. Take a moment to recognize that.', ['course']),
 ('cancelled', 'You are disappointed that our plans changed. We can meet on Saturday instead.', ['Saturday']),
 ('moving', 'You are sad about the move. We can keep our call on Friday.', ['Friday']),
 ('misunderstood', 'You feel hurt by the misunderstanding. I meant that the schedule could change, not that you were excluded.', ['schedule']),
 ('boundary', 'You are disappointed, but I cannot take on the extra shift.', ['cannot', 'shift']),
 ('gift', 'You are pleased that the gift went over well. Tell your friend you are glad they enjoyed it.', ['gift']),
 ('booking', 'You are anxious because the travel plans are uncertain. Confirm the booking details.', ['booking']),
 ('delivery', 'The delay is frustrating for you. I can check the delivery status tomorrow.', ['tomorrow']),
 ('repair', 'You waited all afternoon and feel worn out. Contact the repair service for a new appointment.', ['repair']),
 ('notebook', 'You are upset about losing the notebook. Check the last place you used it.', ['notebook']),
 ('budget', 'The smaller budget is stressful for you. List the essential expenses first.', ['expenses']),
 ('unpacking', 'You feel overwhelmed by the boxes. Start with one essential box.', ['box']),
 ('story', 'You are discouraged about the ending. Try drafting one possible ending without committing to it.', ['ending']),
 ('race', 'You are disappointed that you did not finish the race. Think about what made finishing difficult.', ['race']),
 ('art_group', 'You feel awkward joining the art group. You can introduce yourself to one participant.', ['participant']),
 ('family', 'You are sad about having less time with your family. Set aside a specific time together.', ['family']),
 ('missed_call', 'You feel bad about missing the call. Call back when you are available.', ['call']),
 ('decision', 'You are worried about choosing badly. Write down one priority for the decision.', ['priority']),
 ('volunteering', 'You are pleased that volunteering went well. Check when the next session is scheduled.', ['session']),
 ('refund', 'The unexpected charge is worrying you. I can review it, but I cannot promise a refund.', ['cannot', 'refund']),
 ('caregiving', 'Caring for your father has left you tired. We can move our call to Tuesday.', ['father', 'Tuesday']),
 ('revision_limit', 'You are frustrated with the revisions, but I can review only one more version.', ['one', 'version']),
 ('missed_event', 'You are upset that I missed your event. I was unable to leave work.', ['event', 'work']),
 ('noise', 'The noise has been interrupting your sleep. I will keep the music down after 9 PM.', ['9 PM', 'music']),
 ('distance', 'Being far from home is lonely for you. We can talk this evening.', ['evening']),
 ('application', 'Waiting for the application decision is stressful for you. I do not have an update yet.', ['not', 'update']),
 ('payroll', 'The payment delay is causing you concern. I will contact payroll today.', ['payroll', 'today']),
 ('leave_request', 'You are disappointed, but I cannot approve leave for Friday.', ['cannot', 'Friday']),
 ('accessibility', 'Reading the small text is difficult for you. I can send a larger-print version.', ['version']),
 ('learning', 'You are finding this software confusing. We can go through the instructions together.', ['instructions']),
 ('shared_credit', 'You feel your work was not recognized. I will include your contribution in the report.', ['report']),
 ('broken_plan', 'You are frustrated by the change. The appointment is now on Monday, not Thursday.', ['Monday', 'Thursday', 'not']),
 ('limited_help', 'The situation is difficult for you, but I can help only with the first section.', ['only', 'first', 'section']),
 ('uncertain_result', 'You are worried about the outcome. I cannot tell you yet whether the request will be approved.', ['cannot', 'yet']),
 ('success', 'You worked hard and are pleased with the result. We can share the news with the team.', ['team']),
 ('waiting', 'You have been waiting and are getting frustrated. I will return your call by 4 PM.', ['4 PM']),
 ('competing_needs', 'Your schedule is tight, but I also need time to prepare. Please send the notes today.', ['notes', 'today']),
 ('disagreement', 'You feel strongly about the proposal. I still disagree with the budget change.', ['disagree', 'budget']),
 ('small_step', 'You are feeling uncertain about starting. We can work on the introduction first.', ['introduction']),

]
ORIGINALS = {key: text for key, text, terms in CASES}
SCENARIOS = [dict(id=key, family=key,
    task=('Rewrite the supplied original_message, do NOT reply to it. Return JSON with low copied '
          'verbatim from original_message and high containing only its more empathetic rewrite.'),
    context=json.dumps(dict(original_message=text)), required_terms=terms)
    for key, text, terms in CASES]
VARIANTS = 1

if not RUN.exists():
    make_plan(FEATURE, SCENARIOS, RUN, seed=42, variants=VARIANTS)
manifest, requests = load_plan(RUN)
expected = dict(feature=FeatureDefinition.model_validate(FEATURE).model_dump(),
                scenarios=SCENARIOS, seed=42, variants=VARIANTS)
assert manifest['experiment_id'] == sha(expected), 'Inputs changed: use a fresh RUN and feature version.'
print('Planned pairs:', len(requests), '| splits:', manifest['splits'])
print('Example original:', ORIGINALS[requests[0]['scenario_id']])
print('Teacher task:', SCENARIOS[0]['task'])

# Preserve genuine teacher responses from the earlier pilot, without spending again.
# New experiment IDs differ because the source collection has expanded.
# Reuse is allowed only for byte-identical system/input prompts and generated rows.
def reuse_previous_pilot(previous_run, current_run, current_manifest, current_requests):
    previous_run, current_run = Path(previous_run), Path(current_run)
    source = previous_run / 'responses.jsonl'
    destination = current_run / 'responses.jsonl'
    if destination.exists() or not source.exists():
        return 0
    old_manifest, old_requests = load_plan(previous_run)
    old_by_id = {r['request_id']:r for r in old_requests}
    new_by_prompt = {(r['system'], r['input']):r for r in current_requests}
    imported = []
    for row in read_jsonl(source):
        old_request = old_by_id.get(row.get('request_id'))
        if (old_request is None or row.get('experiment_id') != old_manifest['experiment_id']
                or row.get('status') != 'generated'):
            continue
        matched = new_by_prompt.get((old_request['system'], old_request['input']))
        if matched is None:
            continue
        imported.append(dict(row, request_id=matched['request_id'],
            experiment_id=current_manifest['experiment_id'],
            reused_from=dict(experiment_id=old_manifest['experiment_id'], request_id=row['request_id'],
                             responses_sha256=sha(source.read_bytes()))))
    if len({r['request_id'] for r in imported}) != len(imported):
        raise ValueError('Duplicate previous response IDs; review the previous pilot before reuse.')
    if imported:
        # Existing teacher settings are public configuration, not credentials.
        teacher_file = previous_run / 'teacher.json'
        if not teacher_file.exists():
            raise ValueError('Previous responses have no teacher settings; cannot safely resume.')
        teacher = json.loads(teacher_file.read_text())
        if any(row.get('teacher') != teacher for row in imported):
            raise ValueError('Previous responses and saved teacher settings differ.')
        from secondthought.lab.synthetic import write_jsonl
        (current_run / 'teacher.json').write_text(json.dumps(teacher, indent=2)+'\n')
        write_jsonl(destination, imported)
    return len(imported)

reused = reuse_previous_pilot(RUN.parent / 'empathy-rewrite-v1', RUN, manifest, requests)
print('Reused previous teacher pairs:', reused)
recorded = read_jsonl(RUN / 'responses.jsonl') if (RUN / 'responses.jsonl').exists() else []
print('Recorded responses:', len(recorded), '| remaining requests:', len(requests)-len(recorded))


Planned pairs: 50 | splits: {'extraction': 30, 'test': 10, 'heldout': 10}
Example original: Your workload is heavy, but I need the draft by 3 PM.
Teacher task: Rewrite the supplied original_message, do NOT reply to it. Return JSON with low copied verbatim from original_message and high containing only its more empathetic rewrite.
Reused previous teacher pairs: 0
Recorded responses: 3 | remaining requests: 47


## 2. Enter your API key and choose the teacher model

Run the next cell. It will:

1. Open a **hidden input** for your full OpenAI secret key (`sk-…`). Paste only the key, not Python code or the tracking ID. Press Enter.
2. Check authentication and list candidate text-model IDs accessible to your key.
3. Ask you to paste a model ID from that list. You can also set `TEACHER_MODEL` directly in the cell.

Choose a text model that supports the Responses API. Listing models does not generate pairs. Your key stays in kernel memory/environment and is never printed or saved by this notebook. Enter it again after restarting the kernel. Do not write the literal key into a code cell.

If this run already has teacher settings, the cell reuses its model by default; changing teacher models requires a fresh `RUN` to keep the dataset provenance consistent.

In [21]:
# SETUP ONLY: this cell does not generate any pairs.
from getpass import getpass
from openai import OpenAI, AuthenticationError, PermissionDeniedError, APIConnectionError, APIStatusError

PROVIDER = 'responses'
BASE_URL = None
TEACHER_MODEL = os.environ.get('SYNTHETIC_TEACHER_MODEL', '').strip()  # Or enter a model ID here.
TEACHER_READY = False  # Reset on every attempt; a failed setup cannot reuse old readiness.

_api_key = getpass('Paste ONLY your full OpenAI secret key (input hidden): ').strip()
if not _api_key.startswith('sk-') or any(ch.isspace() for ch in _api_key) or len(_api_key) < 20:
    del _api_key
    raise ValueError('Expected the full secret key beginning sk-. Paste only the key, without quotes, Python code or the tracking ID.')

# Clear a previously malformed environment value and build a new client.
os.environ['OPENAI_API_KEY'] = _api_key
client = OpenAI(api_key=_api_key, max_retries=0, timeout=30)
del _api_key
try:
    available_model_ids = sorted(model.id for model in client.models.list().data)
except AuthenticationError:
    raise RuntimeError('Authentication failed. Rerun this cell with the full active secret key; the shortened key displayed in the dashboard will not work.') from None
except PermissionDeniedError:
    raise RuntimeError('This key cannot list models. Check its project/model permissions in the OpenAI dashboard, then rerun this cell.') from None
except APIConnectionError:
    raise RuntimeError('Could not connect to OpenAI. Check your internet connection, then rerun this cell.') from None
except APIStatusError as exc:
    raise RuntimeError(f'OpenAI model lookup failed (HTTP {exc.status_code}). Check account access and retry this setup cell.') from None

# A shorter candidate list avoids burying the model names among embeddings/audio models.
# Access in this list is not a guarantee of Responses API compatibility.
candidates = [name for name in available_model_ids
              if name.startswith(('gpt-', 'o1', 'o3', 'o4'))
              and not any(term in name for term in ('audio', 'realtime', 'transcribe', 'tts', 'image'))]
print('Authentication succeeded. Candidate text-model IDs:')
for name in (candidates or available_model_ids):
    print(' ', name)

saved_teacher = RUN / 'teacher.json'
if not TEACHER_MODEL and saved_teacher.exists():
    TEACHER_MODEL = json.loads(saved_teacher.read_text())['model']
    print('Resuming the saved teacher:', TEACHER_MODEL)
if not TEACHER_MODEL:
    TEACHER_MODEL = input('Paste the teacher model ID from the list above: ').strip()
if TEACHER_MODEL not in available_model_ids:
    raise ValueError('That model ID was not listed for this key. Rerun setup and enter an exact model ID from the list.')
TEACHER_READY = True
print('Ready. Teacher model:', TEACHER_MODEL)
print('Next: run the generation cell below to finish the 50-pair pilot.')


Authentication succeeded. Candidate text-model IDs:
  gpt-4.1
  gpt-4.1-2025-04-14
  gpt-4.1-mini
  gpt-4.1-mini-2025-04-14
  gpt-4o
  gpt-4o-2024-08-06
  gpt-4o-2024-11-20
  gpt-4o-mini
  gpt-4o-mini-2024-07-18
  gpt-5
  gpt-5-2025-08-07
  gpt-5-mini
  gpt-5-mini-2025-08-07
  gpt-5-nano
  gpt-5-nano-2025-08-07
  gpt-5-pro
  gpt-5-pro-2025-10-06
  gpt-5-search-api
  gpt-5-search-api-2025-10-14
  gpt-5.1
  gpt-5.1-2025-11-13
  gpt-5.2
  gpt-5.2-2025-12-11
  gpt-5.2-pro
  gpt-5.2-pro-2025-12-11
  gpt-5.3-codex
  gpt-5.4
  gpt-5.4-2026-03-05
  gpt-5.4-mini
  gpt-5.4-mini-2026-03-17
  gpt-5.4-nano
  gpt-5.4-nano-2026-03-17
  gpt-5.4-pro
  gpt-5.4-pro-2026-03-05
  gpt-5.5
  gpt-5.5-2026-04-23
  gpt-5.5-pro
  gpt-5.5-pro-2026-04-23
  gpt-5.6-luna
  gpt-5.6-sol
  gpt-5.6-terra
  gpt-6-astra
  gpt-6-luna
  gpt-6-sol
  gpt-6.1-sol
  gpt-6.1-terra
  gpt-6.2-astra
  gpt-6.2-sol
  gpt-live-1
  gpt-live-1.1
  o3
  o3-2025-04-16
Resuming the saved teacher: gpt-4.1
Ready. Teacher model: gpt-4.1
Next:

## 3. Generate the remaining pairs (50 total originals)

This cell makes paid teacher calls, bounded by the number of planned requests. It resumes unrecorded requests, including the 47 remaining after the earlier three-pair pilot. Existing successful pairs are not charged again.

Keep the saved teacher settings for a consistent dataset. Recorded errors are not silently retried. Inspect the output and errors before preparation. Both endpoints must preserve the original practical content; 50 requests do not guarantee 50 accepted pairs.

The teacher copies the original into `low` and writes its empathetic rewrite in `high`. No Qwen weights or vectors are involved yet.

In [22]:
# GENERATION: run only after the setup cell says Ready.
if not globals().get('TEACHER_READY', False):
    raise RuntimeError('Run the API key/model setup cell above successfully before generation.')
MAX_CALLS = len(requests)  # Upper bound per invocation; already-recorded requests are skipped.
# Retain the existing teacher budget when resuming the earlier pilot.
_teacher_path = RUN / 'teacher.json'
MAX_OUTPUT_TOKENS = (json.loads(_teacher_path.read_text())['max_output_tokens']
                     if _teacher_path.exists() else 4096)
result = generate(RUN, PROVIDER, TEACHER_MODEL, max_requests=MAX_CALLS,
                  max_output_tokens=MAX_OUTPUT_TOKENS, base_url=BASE_URL)
display(result)


1: c504653971 generated
2: 8c1253dbb5 generated
3: 822ae8641a generated
4: 881c26a7c6 generated
5: b7957865fd generated
6: bb5cb6daae generated
7: 9a4e04b398 generated
8: cac1307de7 generated
9: 0266a07f74 generated
10: c2c9df1108 generated
11: 9ad7ec8226 generated
12: 993f2d4335 generated
13: 7b5634db29 generated
14: b150de5809 generated
15: 45c2d8d83d generated
16: 0522946e0f generated
17: 23e7453323 generated
18: f3d83687ae generated
19: 257a867138 generated
20: d98cd2a97a generated
21: ac26518ff9 generated
22: 7d87e23523 generated
23: 9417811704 generated
24: 305f40bb3d generated
25: 62648a9390 generated
26: e301712b48 generated
27: c568b846bb generated
28: 6c74c100fe generated
29: 9355391987 generated
30: a2a975d526 generated
31: f36501a7db generated
32: 5ea354a19e generated
33: 11e7c3bdfb generated
34: 0bf2d33430 generated
35: ccbe5c5ebd generated
36: 8842acf919 generated
37: 6c5a6aaf08 generated
38: d55a8df1f3 generated
39: b465497ce5 generated
40: 279eec27b5 generated
41: 0d8f9

{'issued_requests': 47, 'already_recorded': 3, 'pending': 0}

## 4. Inspect original versus rewrite

Verify that these are the **same message**, not a prompt and answer. Reject changes to facts, advice, speaker, promises, refusals or certainty. More polite or longer wording alone is not evidence of empathy. The source-copy check below rejects any teacher attempt to change the original endpoint.

In [23]:
RESPONSES = RUN / 'responses.jsonl'
assert RESPONSES.exists(), 'Run the generation cell first.'
from secondthought.lab.synthetic import parse_contrast, write_jsonl
raw_rows = read_jsonl(RESPONSES)
request_by_id = {r['request_id']:r for r in requests}

for row in raw_rows:
    request = request_by_id.get(row.get('request_id'))
    if request is None or row.get('experiment_id') != manifest['experiment_id']:
        raise ValueError('Response does not belong to this rewrite plan.')
    original = ORIGINALS[request['scenario_id']]
    print('Request:', row['request_id'], '|', row.get('status'))
    print('ORIGINAL:', original)
    if row.get('status') != 'generated':
        print('Generation error:', row.get('error_type'), '\n')
        continue
    try:
        pair = parse_contrast(row['output'])
        print('REWRITE: ', pair.high)
        print('Source copied exactly:', pair.low == original, '\n')
    except (ValueError, TypeError):
        print('Invalid pair JSON:', row.get('output'), '\n')


Request: 96ad8fc46505a6f52cc03dd970afd12d866a888097fe046614103fd524b8611b | generated
ORIGINAL: Your workload is heavy, but I need the draft by 3 PM.
REWRITE:  I see your workload is heavy, but I still need the draft by 3 PM.
Source copied exactly: True 

Request: 05811564d665df2d59126615c83761f238a4358633416c2efa5714d704f9e2a6 | generated
ORIGINAL: You are worried about the meeting. Let us review the agenda together.
REWRITE:  I understand you are worried about the meeting. Let us review the agenda together.
Source copied exactly: True 

Request: b79c6b798b1a19b8944a3659ede9f8c2e9f1a1f581c62b700bb9e09295753f3e | generated
ORIGINAL: You feel overlooked because your contribution is missing from the notes. Ask for it to be added.
REWRITE:  You feel overlooked because your contribution is missing from the notes. Request that it be added so your effort is recognized.
Source copied exactly: True 

Request: c5046539716b1697b1ce9bb64569f3518cbb50809e7f21ff65e7228feedf72cc | generated
ORIGINAL

## 5. Validate the anchored pairs and export

Pairs whose `low` is not the exact original are excluded and logged in `source_copy_rejections.jsonl`. Shared checks then reject duplicates, changed numbers, missing required terms and large length differences. **These checks do not establish semantic equivalence or empathy. Review the outputs manually.**

Set `APPROVALS` to a JSON list of reviewed request IDs to retain only reviewed pairs. Use a fresh `DATASET_DIR` after changing responses or approvals. The cell now detects changed input/review files instead of silently exporting a stale prepared dataset.

CSV columns are `original` and `empathetic_rewrite`. JSONL uses the existing pipeline's `source` and `target` names. The CSV includes a split column; reserved test rows remain excluded from extraction `pairs.jsonl`.

In [24]:
APPROVALS = None  # Optional: RUN / 'approved_request_ids.json'
DATASET_DIR = RUN / 'dataset'
assert RESPONSES.exists(), 'Generate the rewrites first.'
request_by_id = {r['request_id']:r for r in requests}
filtered, source_rejections = [], []
for row in read_jsonl(RESPONSES):
    request = request_by_id.get(row.get('request_id'))
    if request is None or row.get('experiment_id') != manifest['experiment_id']:
        raise ValueError('Response is not bound to this rewrite experiment.')
    if row.get('status') == 'generated':
        try:
            pair = parse_contrast(row['output'])
        except (ValueError, TypeError):
            pair = None  # Shared preparation records the schema error.
        if pair is not None and pair.low != ORIGINALS[request['scenario_id']]:
            source_rejections.append(dict(request_id=row['request_id'], reason='original_was_changed'))
            row = dict(row, status='error', error_type='OriginalWasChanged')
    filtered.append(row)

# Fingerprint raw inputs and review decisions before reusing prepared data.
binding = dict(responses_sha256=sha(RESPONSES.read_bytes()),
               approvals_sha256=sha(Path(APPROVALS).read_bytes()) if APPROVALS else None)
state = DATASET_DIR / 'notebook_inputs.json'
if (DATASET_DIR / 'pairs.jsonl').exists():
    if not state.exists() or json.loads(state.read_text()) != binding:
        raise ValueError('Responses or approvals changed. Choose a fresh DATASET_DIR; saved pairs are immutable.')
    summary = json.loads((DATASET_DIR / 'summary.json').read_text())
    print('Reusing the matching saved dataset.')
else:
    anchored_path = RUN / 'anchored_responses.jsonl'
    write_jsonl(anchored_path, filtered)
    write_jsonl(RUN / 'source_copy_rejections.jsonl', source_rejections)
    summary = prepare(RUN, responses=anchored_path, output=DATASET_DIR, approvals=APPROVALS)
    state.write_text(json.dumps(binding, indent=2)+'\n')
display(summary)
print('Original-copy failures:', len(source_rejections))
accepted = read_jsonl(DATASET_DIR / 'accepted.jsonl')
with (DATASET_DIR / 'empathy_pairs.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=['row_id', 'split', 'original', 'empathetic_rewrite'])
    writer.writeheader()
    writer.writerows(dict(row_id=p['row_id'], split=p['split'], original=p['source'],
                         empathetic_rewrite=p['target']) for p in accepted)
print('CSV:', DATASET_DIR / 'empathy_pairs.csv')
print('Accepted:', DATASET_DIR / 'accepted.jsonl')
print('Rejected:', DATASET_DIR / 'rejected.jsonl')
print('Source-copy failures:', RUN / 'source_copy_rejections.jsonl')
print('Reserved test:', DATASET_DIR / 'evaluation.jsonl')
print('Pairs saved. Continue below to extract vectors and build the test website catalog.')


{'experiment_id': '0fbb7df878f5c11d7a83b721421b1582cc42e6b0cf38e36cd417c374fada5c0a',
 'accepted': 50,
 'rejected': 0,
 'pending': 0,
 'split_counts': {'extraction': 30, 'test': 10, 'heldout': 10},
 'rejection_reasons': {},
 'ready_for_extraction': True,
 'review': 'heuristics_only_unreviewed'}

Original-copy failures: 0
CSV: /Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/dataset/empathy_pairs.csv
Accepted: /Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/dataset/accepted.jsonl
Rejected: /Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/dataset/rejected.jsonl
Source-copy failures: /Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/source_copy_rejections.jsonl
Reserved test: /Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/dataset/evaluation.jsonl
Pairs saved. Continue below to extract vectors and build the test website catalog.


## 6. Extract the 50-pair pilot using our current Qwen pipeline

Run after inspecting and preparing all 50 pairs. This deliberately stops if any pair is pending/rejected; inspect the rejection logs and correct the responses, then prepare into a fresh `DATASET_DIR` before continuing.

The existing extractor uses **30 extraction pairs**, measures ordering on **10 held-out pairs**, and leaves **10 reserved test pairs** untouched. It produces **48 vectors: 24 decoder blocks × mean/final-token pooling**. The raw direction is the average empathetic-rewrite activation minus original-message activation. Inference uses unit-L2 scaling, matching existing empathy.

The cell loads the pinned Qwen model from the local cache and may take a few minutes. It does not generate teacher pairs or modify the production catalog. Existing artifacts are validated and reused rather than overwritten.

In [25]:
import subprocess
import torch
from research.steering_experimentation.extract_lab_vectors import load_prepared_pairs

PAIRS = DATASET_DIR / 'pairs.jsonl'
summary = json.loads((DATASET_DIR / 'summary.json').read_text())
if summary['accepted'] != 50 or summary['pending'] != 0 or summary['rejected'] != 0:
    raise RuntimeError('Need all 50 accepted pairs before extraction. Finish generation and review the rejection logs, then re-prepare if needed.')
if summary['split_counts'] != dict(extraction=30, heldout=10, test=10):
    raise RuntimeError('Unexpected split: expected 30 extraction / 10 held-out / 10 reserved test.')
extraction, heldout, metadata = load_prepared_pairs(PAIRS, 'empathy')
assert len(extraction)==30 and len(heldout)==10
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
ARTIFACT = RUN / f"vectors/qwen2_5_0_5b_empathy_all_layers_{FEATURE['version']}.pt"
if not ARTIFACT.exists():
    subprocess.run([sys.executable, '-m', 'research.steering_experimentation.synthetic_pipeline',
        'extract', '--run', str(RUN), '--dataset', str(PAIRS), '--device', DEVICE,
        '--batch-size', '8', '--max-length', '256'], check=True)
artifact = torch.load(ARTIFACT, map_location='cpu', weights_only=True)
if (artifact['provenance']['experiment_id'] != manifest['experiment_id']
        or artifact['provenance']['pairs_sha256'] != sha(PAIRS.read_bytes())):
    raise ValueError('Artifact does not match this prepared dataset. Use a fresh run/version.')
assert len(artifact['raw_directions'])==48
print('Ready:', ARTIFACT)
print('Extracted:', len(artifact['raw_directions']), 'vectors; device:', DEVICE)
for pooling in ('mean', 'final_token'):
    print(pooling, {i:round(artifact['candidates'][f'layer{i}_{pooling}']['heldout_ordering_accuracy'],2)
                    for i in range(24)})


/Users/nandanprince/Desktop/NLP_project/SecondThought/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


empathy: validated 30/10 unique, disjoint source pairs
Loaded pinned Qwen/Qwen2.5-0.5B-Instruct@7ae557604adf67be50417f59c2c2f167def9a775, 24 layers on mps
empathy extraction 8/30 pairs; 0.3s
empathy extraction 30/30 pairs; 0.8s
empathy heldout 8/10 pairs; 0.9s
empathy heldout 10/10 pairs; 1.0s
Saved /Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/vectors/qwen2_5_0_5b_empathy_all_layers_rewrite-v1.pt: 48 vectors; sha256=ed0461434a4429c1433a300de27976652e2b0d0b6445280917d12b96775a02e7
Ready: /Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/vectors/qwen2_5_0_5b_empathy_all_layers_rewrite-v1.pt
Extracted: 48 vectors; device: mps
mean {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0, 10: 1.0, 11: 1.0, 12: 1.0, 13: 1.0, 14: 1.0, 15: 1.0, 16: 1.0, 17: 1.0, 18: 1.0, 19: 1.0, 20: 1.0, 21: 1.0, 22: 1.0, 23: 0.9}
final_token {0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0, 5: 1.0, 6: 1.0, 7

## 7. Connect the pilot to the website controls

This builds a **separate test catalog**, replacing only its empathy vectors. Existing formality and other features are retained. The website's empathy checkbox, pooling selector, all 24 layers, per-layer alpha, multi-layer mixing, and measured layer colours resolve to these new vectors through the existing backend. Ten ordering pairs give scores in 10% increments; colours do not measure generation quality.

The normal website and catalog stay available as the original-vector baseline. No production promotion is performed.

In [26]:
from secondthought.lab.catalog import Catalog
from secondthought.lab.experiments import publish_experiment

BASE = Catalog.load(ROOT / 'artifacts/catalog/qwen2_5_0_5b.json', ROOT / 'artifacts')
BASE_VERSION = BASE.version
SERVING = RUN / 'serving'
if not SERVING.exists():
    display(publish_experiment(RUN, ARTIFACT, SERVING, base=BASE))
EXPERIMENT = Catalog.load(SERVING / 'catalog/qwen2_5_0_5b.json', SERVING)
assert not EXPERIMENT.issues, EXPERIMENT.issues
empathy_vectors = [v for v in EXPERIMENT.spec.vectors if v.feature_id=='empathy']
assert len(empathy_vectors)==48
assert len({(v.layer_index, v.pooling) for v in empathy_vectors})==48
for entry in empathy_vectors:
    EXPERIMENT.vector(entry.id)  # Validate each applied direction.
expected_hash = sha(ARTIFACT.read_bytes())
assert all(EXPERIMENT.artifacts[v.artifact_id].sha256 == expected_hash for v in empathy_vectors)
assert Catalog.load(ROOT / 'artifacts/catalog/qwen2_5_0_5b.json', ROOT / 'artifacts').version==BASE_VERSION
print('Website experiment catalog ready:', EXPERIMENT.version)
# Save reproducible start commands without credentials.
import shlex
commands = ('cd '+shlex.quote(str(ROOT))+'\n'
    'export SECONDTHOUGHT_LAB_CATALOG='+shlex.quote(str(SERVING / 'catalog/qwen2_5_0_5b.json'))+'\n'
    'export SECONDTHOUGHT_LAB_ARTIFACT_ROOT='+shlex.quote(str(SERVING))+'\n'
    '.venv/bin/python -m website.api.check_setup\n'
    '.venv/bin/python -m uvicorn website.api.server:app --host 127.0.0.1 --port 8004 --workers 1\n')
(RUN / 'start_website.sh').write_text(commands)
print('Run these commands in a separate terminal:\n\n'+commands)
print('Open http://127.0.0.1:8004/steering')


{'catalog_version': 'e22c559a7072854da944daf4298b21af25c0de8258b2736c37b25196aa6bfd25',
 'feature': 'empathy',
 'artifact_id': 'qwen-empathy-synthetic-rewrite-v1',
 'vectors': 48,
 'artifact_root': '/Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/serving',
 'catalog': '/Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/serving/catalog/qwen2_5_0_5b.json'}

Website experiment catalog ready: e22c559a7072854da944daf4298b21af25c0de8258b2736c37b25196aa6bfd25
Run these commands in a separate terminal:

cd /Users/nandanprince/Desktop/NLP_project/SecondThought
export SECONDTHOUGHT_LAB_CATALOG=/Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/serving/catalog/qwen2_5_0_5b.json
export SECONDTHOUGHT_LAB_ARTIFACT_ROOT=/Users/nandanprince/Desktop/NLP_project/SecondThought/data/synthetic/empathy-rewrite-50-v1/serving
.venv/bin/python -m website.api.check_setup
.venv/bin/python -m uvicorn website.api.server:app --host 127.0.0.1 --port 8004 --workers 1

Open http://127.0.0.1:8004/steering


## 8. Check empathy steering

Enable only Empathy first. Use a fixed task and system prompt, choose one layer and pooling, then compare alpha **0, +0.5, +1, +2, and -1**. Keep the input unchanged. Repeat on reserved originals from `dataset/evaluation.jsonl` and fresh messages. Compare with the normal website to assess the original empathy vectors.

Check: acknowledgement of the recipient, unchanged facts/actions/stance, unsupported reassurance, repetition, and coherence. Then try two modest per-layer strengths in multi-layer mode. A higher held-out ordering score alone does not establish better generated rewrites. This 50-pair run tests the complete integration and gives an initial behavioural check, not a reliability claim.